In [ ]:
!pip install catboost

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# ===============================
# 1. Load data
# ===============================
df = pd.read_csv("/content/pet-records.csv")  # Change to your dataset path

# ===============================
# 2. Handle missing values
# ===============================
for col in ['petAge', 'petGender', 'petBreed', 'Note']:
    df[col] = df[col].fillna("Unknown")

# ===============================
# 3. Feature Engineering
# ===============================

def age_to_months(age_str):
    if isinstance(age_str, str):
        age_str = age_str.lower()
        years = sum(int(x) for x in re.findall(r'(\d+)\s*year', age_str))
        months = sum(int(x) for x in re.findall(r'(\d+)\s*month', age_str))
        if years == 0 and months == 0:
            nums = re.findall(r'\d+', age_str)
            if nums:
                years = int(nums[0])
        return years * 12 + months
    return np.nan

df['petAgeMonths'] = df['petAge'].apply(age_to_months)

breed_freq = df['petBreed'].value_counts().to_dict()
df['breed_freq'] = df['petBreed'].map(breed_freq)
df['is_mix_breed'] = df['petBreed'].str.lower().str.contains("mix|cross").astype(int)
df['PetCancerSuspect'] = df['PetCancerSuspect'].map({"Yes": 1, "No": 0}).fillna(0)

df['note_length'] = df['Note'].apply(lambda x: len(str(x)))
df['note_word_count'] = df['Note'].apply(lambda x: len(str(x).split()))
keywords = ['tumor', 'swelling', 'cancer', 'mass', 'lump']
df['note_keyword_count'] = df['Note'].apply(lambda x: sum(kw in str(x).lower() for kw in keywords))

gender_ohe = pd.get_dummies(df['petGender'], prefix='gender')
df = pd.concat([df, gender_ohe], axis=1)

# ===============================
# 4. Prepare training data
# ===============================
features = [
    'petAgeMonths',
    'breed_freq',
    'is_mix_breed',
    'PetCancerSuspect',
    'note_length',
    'note_word_count',
    'note_keyword_count'
] + list(gender_ohe.columns)

X = df[features]
y = df['PetRiskValue']
X = X.fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ===============================
# 5. Train and evaluate traditional ML models
# ===============================
models = {
    "Linear Regression": LinearRegression(),
    "ElasticNet": ElasticNet(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "Bagging Regressor": BaggingRegressor(),
    "KNN": KNeighborsRegressor(),
    "XGBoost": XGBRegressor(eval_metric='rmse', verbosity=0),
    "LightGBM": LGBMRegressor(verbose=-1),
    "CatBoost": CatBoostRegressor(verbose=0)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2 = r2_score(y_test, preds)
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    results.append({
        "Model": name,
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse
    })

# ===============================
# 6. Deep Learning Model
# ===============================
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

dl_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1)  # regression output
])

dl_model.compile(optimizer='adam', loss='mse')

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

dl_model.fit(
    X_train_scaled, y_train,
    validation_data=(X_test_scaled, y_test),
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

dl_preds = dl_model.predict(X_test_scaled).flatten()
r2_dl = r2_score(y_test, dl_preds)
mae_dl = mean_absolute_error(y_test, dl_preds)
mse_dl = mean_squared_error(y_test, dl_preds)
rmse_dl = np.sqrt(mse_dl)

results.append({
    "Model": "Deep Learning (Keras)",
    "MAE": mae_dl,
    "MSE": mse_dl,
    "RMSE": rmse_dl
})

# ===============================
# 7. Show results
# ===============================
results_df = pd.DataFrame(results).sort_values(by="R²", ascending=False)
print("\nModel Comparison:")
print(results_df.to_string(index=False))

